# Python lab: Introduction to Numerical Methods

ใช้ Python standard library และกด **Run All** ตามลำดับได้โดยไม่ดาวน์โหลดข้อมูลหรือไฟล์โค้ดเพิ่มเติม ภาพประกอบฝังอยู่ใน Notebook แล้ว เนื้อหาอ่านจาก `numerical-methods.md` ตัวเลขทั้งหมดเป็นตัวอย่างสมมติของ European Call/Put ภายใต้ GBM ที่มีค่าพารามิเตอร์คงที่ ไม่มีเงินปันผลและต้นทุนซื้อขาย

ตัวสุ่ม Python ใช้ `random.Random(seed).gauss` จึงทำซ้ำได้แยกจากตัวสุ่มบนเว็บไซต์ Seed เลขเดียวกันไม่รับประกันผลตรงกันข้ามภาษา ช่วงความเชื่อมั่นครอบคลุม sampling error ภายใต้แบบจำลอง ส่วน finite difference มี discretization และ finite-domain error

[เปิดบทเรียน](https://nutdnuy.github.io/quantitative-finance-notes/numerical-methods.html)

# Introduction to Numerical Methods

ถ้าไม่มีสูตรสำเร็จ เราจะหาราคา Option และรู้ได้อย่างไรว่าคำตอบแม่นพอ?

ในบท [Black–Scholes Model](https://nutdnuy.github.io/quantitative-finance-notes/black-scholes-model.html) เราได้สูตรราคาของ European Call และ Put ภายใต้สมมติฐานที่กำหนดไว้ แต่เมื่อเปลี่ยน payoff ให้ขึ้นกับหลายสินทรัพย์หรือราคาตลอดเส้นทาง สูตรปิดอาจหาได้ยาก เราจึงต้องเปลี่ยนความสัมพันธ์ทางคณิตศาสตร์ให้เป็นขั้นตอนคำนวณ

บทนี้ใช้สองมุมมองของปัญหาเดียวกัน: **Monte Carlo** ประมาณค่าคาดหมายของ payoff ส่วน **finite difference** ประมาณคำตอบของสมการอนุพันธ์บนกริด เราจะเริ่มจากสัญญาที่มีสูตรราคาอยู่แล้ว เพื่อใช้สูตรนั้นตรวจวิธีคำนวณก่อนนำไปใช้กับโจทย์ที่ซับซ้อนขึ้น

บทก่อนหน้า [Asset Returns — Empirical Stylized Facts](https://nutdnuy.github.io/quantitative-finance-notes/asset-returns-stylized-facts.html) ชวนตรวจว่าแบบจำลองเหมาะกับข้อมูลหรือไม่ บทนี้ถามอีกชั้นว่า **เมื่อเลือกแบบจำลองแล้ว เราคำนวณคำตอบของมันถูกต้องเพียงใด** ตัวเลขที่ลู่เข้าสวยงามยังไม่ได้ยืนยันว่าแบบจำลองอธิบายตลาดได้ดี

ตัวอย่างทั้งหมดเป็นข้อมูลสมมติ ใช้หุ้นไม่มีปันผล European Option หนึ่งหน่วย ราคาและ payoff มีหน่วยดอลลาร์ เวลาเป็นปี ดอกเบี้ยทบต้นต่อเนื่อง ค่าตั้งต้นคือ \(S_0=K=100\), \(r=3\%\) ต่อปี, \(\sigma=20\%\) ต่อรากปี และ \(T=1\) ปี โดย r และ σ คงที่

In [1]:
"""Reproducible European-option examples; Python standard library only.

Constant-parameter GBM, no dividends, no transaction costs, rates in decimals,
time in years. These functions illustrate numerical methods, not market data.
"""
import math
import random
from statistics import NormalDist

NORMAL = NormalDist()
DEFAULT_SEED = 2530401


def _parameters(S, K, r, sigma, T, kind):
    if not all(math.isfinite(v) for v in (S, K, r, sigma, T)):
        raise ValueError("Inputs must be finite")
    if S < 0 or K <= 0 or sigma < 0 or T < 0:
        raise ValueError("Require S>=0, K>0, sigma>=0 and T>=0")
    if kind not in ("call", "put"):
        raise ValueError("kind must be call or put")


def payoff(S, K=100., kind="call"):
    if kind not in ("call", "put"):
        raise ValueError("kind must be call or put")
    return max(S-K, 0.) if kind == "call" else max(K-S, 0.)


def black_scholes(S=100., K=100., r=.03, sigma=.2, T=1., kind="call"):
    """Analytic reference price, with exact zero-time/volatility limits."""
    _parameters(S, K, r, sigma, T, kind)
    discount = math.exp(-r*T)
    if T == 0:
        return payoff(S, K, kind)
    if sigma == 0 or S == 0:
        return discount*payoff(S*math.exp(r*T), K, kind)
    d1 = (math.log(S/K)+(r+.5*sigma*sigma)*T)/(sigma*math.sqrt(T))
    d2 = d1-sigma*math.sqrt(T)
    if kind == "call":
        return S*NORMAL.cdf(d1)-K*discount*NORMAL.cdf(d2)
    return K*discount*NORMAL.cdf(-d2)-S*NORMAL.cdf(-d1)


def black_scholes_greeks(S=100., K=100., r=.03, sigma=.2, T=1., kind="call"):
    """Delta, gamma and calendar-time theta; exclude nonsmooth limits."""
    _parameters(S, K, r, sigma, T, kind)
    if min(S, sigma, T) <= 0:
        raise ValueError("Greeks require S, sigma and T > 0")
    d1 = (math.log(S/K)+(r+.5*sigma*sigma)*T)/(sigma*math.sqrt(T))
    d2 = d1-sigma*math.sqrt(T)
    gamma = NORMAL.pdf(d1)/(S*sigma*math.sqrt(T))
    common = -S*NORMAL.pdf(d1)*sigma/(2*math.sqrt(T))
    if kind == "call":
        delta = NORMAL.cdf(d1)
        theta = common-r*K*math.exp(-r*T)*NORMAL.cdf(d2)
    else:
        delta = NORMAL.cdf(d1)-1
        theta = common+r*K*math.exp(-r*T)*NORMAL.cdf(-d2)
    return {"delta": delta, "gamma": gamma, "theta": theta}


def monte_carlo(S=100., K=100., r=.03, sigma=.2, T=1., kind="call",
                paths=20000, seed=DEFAULT_SEED, checkpoints=()):
    """IID exact GBM terminals, Welford sample variance and approximate 95% CI.

    Python random.Random(seed).gauss draws are reproducible within this example;
    a browser's different RNG need not generate the same sample. All checkpoints
    are prefixes of ONE sample, not independent replications.
    """
    _parameters(S, K, r, sigma, T, kind)
    if isinstance(paths, bool) or not isinstance(paths, int) or paths < 2:
        raise ValueError("paths must be an integer >= 2")
    points = set(checkpoints) | {paths}
    if any(isinstance(n, bool) or not isinstance(n, int) or not 2 <= n <= paths for n in points):
        raise ValueError("Checkpoints must be integers between 2 and paths")
    rng = random.Random(seed)
    mean, m2, terminal_mean = 0., 0., 0.
    discount = math.exp(-r*T)
    drift, diffusion = (r-.5*sigma*sigma)*T, sigma*math.sqrt(T)
    rows = []
    for n in range(1, paths+1):
        terminal = S*math.exp(drift+diffusion*rng.gauss(0., 1.))
        value = discount*payoff(terminal, K, kind)
        delta = value-mean
        mean += delta/n
        m2 += delta*(value-mean)
        terminal_mean += (terminal-terminal_mean)/n
        if n in points:
            sd = math.sqrt(max(m2/(n-1), 0.))
            se = sd/math.sqrt(n)
            rows.append({"paths": n, "price": mean, "sd": sd, "se": se,
                         "lower": mean-1.96*se, "upper": mean+1.96*se})
    return {**rows[-1], "seed": seed, "checkpoints": rows,
            "terminal_mean": terminal_mean,
            "reference": black_scholes(S, K, r, sigma, T, kind)}


def explicit_coefficients(r=.03, sigma=.2, T=1., intervals=80, steps=1000):
    """Return central-difference weights after a sufficient monotonicity check.

    This teaching implementation requires nonnegative r and all three weights.
    More time steps repair negative b; negative a due to r>sigma^2 at i=1
    requires a different drift stencil/model choice, not just a smaller dt.
    """
    if not all(math.isfinite(v) for v in (r, sigma, T)) or r < 0 or sigma <= 0 or T <= 0:
        raise ValueError("Require finite r>=0, sigma>0, T>0")
    if any(isinstance(v, bool) or not isinstance(v, int) for v in (intervals, steps)):
        raise ValueError("Grid counts must be integers")
    if intervals < 3 or steps < 1:
        raise ValueError("Require intervals>=3 and steps>=1")
    dt = T/steps
    coefficients = [(.5*dt*(sigma*sigma*i*i-r*i),
                     1-dt*(sigma*sigma*i*i+r),
                     .5*dt*(sigma*sigma*i*i+r*i)) for i in range(1, intervals)]
    minimum = min(min(row) for row in coefficients)
    if minimum < 0:
        if sigma*sigma < r:
            raise ValueError("Unsafe central drift: a_1<0; more time steps do not fix it")
        raise ValueError("Unsafe explicit time step: negative coefficient; increase steps")
    return coefficients


def finite_difference(S=100., K=100., r=.03, sigma=.2, T=1., kind="call",
                      Smax=400., intervals=80, steps=1000):
    """Explicit European Black-Scholes solver marching forward in tau=T-t.

    Uniform S grid. Call high boundary uses the large-S asymptote; Put high
    boundary uses zero. Both are finite-domain approximations. Greeks use the
    nearest interior node; returned greek_spot makes that location explicit.
    """
    _parameters(S, K, r, sigma, T, kind)
    if not math.isfinite(Smax) or Smax <= max(S, K):
        raise ValueError("Require finite Smax greater than S and K")
    coefficients = explicit_coefficients(r, sigma, T, intervals, steps)
    dS, dt = Smax/intervals, T/steps
    spots = [i*dS for i in range(intervals+1)]
    values = [payoff(s, K, kind) for s in spots]
    for n in range(1, steps+1):
        previous = values
        discount_strike = K*math.exp(-r*n*dt)
        values = [0.]*(intervals+1)
        values[0] = 0. if kind == "call" else discount_strike
        values[-1] = Smax-discount_strike if kind == "call" else 0.
        for i, (a, b, c) in enumerate(coefficients, 1):
            values[i] = a*previous[i-1]+b*previous[i]+c*previous[i+1]
    index = min(int(S/dS), intervals-1)
    weight = S/dS-index
    price = values[index]*(1-weight)+values[index+1]*weight
    greek_index = min(max(round(S/dS), 1), intervals-1)
    delta = (values[greek_index+1]-values[greek_index-1])/(2*dS)
    gamma = (values[greek_index+1]-2*values[greek_index]+values[greek_index-1])/dS**2
    theta = -(values[greek_index]-previous[greek_index])/dt
    return {"price": price, "reference": black_scholes(S, K, r, sigma, T, kind),
            "spots": spots, "values": values, "dS": dS, "dt": dt,
            "delta": delta, "gamma": gamma, "theta": theta,
            "greek_spot": spots[greek_index], "intervals": intervals, "steps": steps,
            "coefficient_min": min(min(row) for row in coefficients)}


def convergence_rows():
    """Coupled refinements keep dt/dS^2 fixed, with the strike on each grid."""
    rows = []
    for intervals in (40, 80, 160, 320):
        steps = 250*(intervals//40)**2
        result = finite_difference(intervals=intervals, steps=steps)
        rows.append({key: result[key] for key in ("intervals", "steps", "dS", "dt", "price", "reference")}
                    | {"error": abs(result["price"]-result["reference"])})
    return rows


def close(a,b,tolerance=1e-10):
    assert math.isclose(a,b,rel_tol=tolerance,abs_tol=tolerance),(a,b)
print("Loaded self-contained math functions; Python standard library only.")

Loaded self-contained math functions; Python standard library only.


## เริ่มจากราคาที่เราต้องการประมาณ

ภายใต้สมมติฐาน no-arbitrage ของ Black–Scholes เราใช้กระบวนการราคาภายใต้ [risk-neutral measure](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#risk-neutral-measure) \(\mathbb Q\)

$$
dS_t=rS_t\,dt+\sigma S_t\,dW_t^{\mathbb Q}.
$$

ราคา ณ เวลา t ของสัญญาที่จ่าย \(g(S_T)\) เมื่อหมดอายุคือ

$$
V(S_t,t)=e^{-r(T-t)}\mathbb E^{\mathbb Q}[g(S_T)\mid S_t].
$$

สำหรับ Call ใช้ \(g(S_T)=\max(S_T-K,0)\) ส่วน Put ใช้ \(\max(K-S_T,0)\) ต้องแยก **payoff ณ วันหมดอายุ** ออกจาก **ราคาวันนี้** และกำไรหลังหัก premium

เหตุผลที่ drift เป็น r ไม่ใช่ผลตอบแทนคาดหวังจริง μ มาจากการตีราคาที่สอดคล้องกับพอร์ตเลียนแบบและ no-arbitrage ไม่ใช่การทำนายว่าหุ้นจริงจะโตเท่าดอกเบี้ย หรือการสมมติว่าผู้ลงทุนทุกคนไม่กลัวความเสี่ยง ทบทวนที่ [พอร์ตเลียนแบบและ risk-neutral pricing](https://nutdnuy.github.io/quantitative-finance-notes/binomial-model.html)

สำหรับตัวอย่างนี้ สูตร Black–Scholes ให้ Call ประมาณ **9.4134 ดอลลาร์** และ Put ประมาณ **6.4580 ดอลลาร์** เราจะใช้เป็น benchmark ของทั้งสองวิธี ไม่ใช่ราคาตลาดที่สังเกตมา

In [2]:
S0,K,r,sigma,T = 100.,100.,.03,.2,1.
call,put = black_scholes(),black_scholes(kind="put")
close(call,9.413403383853016)
close(put,6.457956738703835)
close(call-put,S0-K*math.exp(-r*T))
print(f"Analytic reference: Call={call:.10f}; Put={put:.10f}")
print(f"Put-call parity: C-P={call-put:.10f}=S0-K*exp(-r*T)")
print("Risk-neutral drift r is a pricing assumption, not an estimate of physical expected returns.")

Analytic reference: Call=9.4134033839; Put=6.4579567387
Put-call parity: C-P=2.9554466451=S0-K*exp(-r*T)
Risk-neutral drift r is a pricing assumption, not an estimate of physical expected returns.


## สุ่มราคาปลายทางในครั้งเดียว

จาก [Itô’s lemma](https://nutdnuy.github.io/quantitative-finance-notes/applied-stochastic-calculus.html) สำหรับ log S เราได้

$$
S_T=S_0\exp\left[\left(r-\frac{\sigma^2}{2}\right)T+\sigma\sqrt T\,Z\right],
\qquad Z\sim N(0,1).
$$

นี่คือ **exact terminal simulation ภายใต้ GBM ที่พารามิเตอร์คงที่** จึงไม่ต้องแบ่งวันเพื่อหาราคา European Call/Put ที่ payoff ขึ้นกับราคาปลายทางเพียงค่าเดียว คำว่า exact หมายถึงไม่มีความคลาดเคลื่อนจากการแบ่งเวลาในสูตรนี้ การเฉลี่ยจากจำนวนตัวอย่างจำกัดยังมีความคลาดเคลื่อนอยู่

เมื่อแทนค่าตั้งต้น ถ้าช็อกหนึ่งรอบเป็น Z=0 จะได้ \(S_T=100e^{0.01}\approx101.0050\) และ discounted Call payoff ประมาณ 0.9753 ดอลลาร์ นี่คือผลจาก **หนึ่งช็อก** ไม่ใช่ราคา Call ทั้งสัญญา

ขั้นตอนคำนวณมีดังนี้

1. สุ่ม \(Z_j\) จาก Standard Normal อย่างอิสระ
2. คำนวณ \(S_T^{(j)}\) ด้วยสูตร exponential
3. คำนวณ discounted payoff \(Y_j=e^{-rT}g(S_T^{(j)})\)
4. ทำซ้ำ N รอบ แล้วเฉลี่ย \(\widehat V_N=N^{-1}\sum_{j=1}^N Y_j\)

**ถ้า payoff ขึ้นกับเส้นทางล่ะ?** Asian Option ที่ใช้ค่าเฉลี่ยราคาตามวันสังเกตต้องจำลองราคาทุกวันที่สัญญาระบุ เรายังใช้ exact GBM update ทีละช่วงได้ แต่การสุ่มเพียง \(S_T\) ไม่ให้ข้อมูลระหว่างทาง สำหรับ barrier ที่เฝ้าราคาต่อเนื่อง แม้จำลองจุดกริดได้ตรงตาม GBM ก็ยังอาจพลาดการข้าม barrier ระหว่างจุด

หาก SDE ไม่มีวิธีจำลอง exact ที่สะดวก อาจใช้ [Euler–Maruyama](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#euler-maruyama)

$$
S_{n+1}=S_n+rS_n\Delta t+\sigma S_n\sqrt{\Delta t}\,Z_n.
$$

สูตร Euler ในระดับราคาอาจให้ราคาติดลบ และมี discretization error ภายใต้เงื่อนไขมาตรฐานที่เหมาะสม weak error ของค่าคาดหมายมีอันดับ \(O(\Delta t)\) ส่วน strong error ที่เทียบเส้นทางด้วย Brownian motion เดียวกันโดยทั่วไปเป็น \(O(\sqrt{\Delta t})\) อัตราเหล่านี้ต้องตรวจเงื่อนไขของ SDE และ payoff ไม่ใช่กฎที่ใช้ได้กับทุกสัญญา [Mike Giles, Lecture 9](https://people.maths.ox.ac.uk/gilesm/mc/mc/lec9.pdf)

เอกสารต้นทางเสนอ \(\sum_{i=1}^{12}U_i-6\) เป็นวิธีประมาณ Normal อย่างง่าย ค่านี้มี mean 0 และ variance 1 แต่มีช่วงจำกัด [−6,6] จึงไม่ใช่ Normal จริง ห้องทดลองใช้ Box–Muller แปลง uniform pseudo-random numbers เป็น Normal แทน และระบุ seed เพื่อให้ทำซ้ำได้ [วิธี Box–Muller](https://people.maths.ox.ac.uk/gilesm/mc/mc/lec1.pdf#page=13)

In [3]:
for z in [-2.,0.,2.]:
    terminal = S0*math.exp((r-.5*sigma*sigma)*T+sigma*math.sqrt(T)*z)
    assert terminal > 0
    close(math.log(terminal/S0),(r-.5*sigma*sigma)*T+sigma*math.sqrt(T)*z)
    print(f"Z={z:+.1f}: exact S_T={terminal:.8f}; Call payoff={payoff(terminal):.8f}")
expected_terminal = S0*math.exp(r*T)
close(math.exp(-r*T)*expected_terminal,S0)
print(f"Analytic E_Q[S_T]={expected_terminal:.8f}; discounted expectation={S0:.8f}")
print("One exact terminal draw has no time-discretization error for this European payoff and constant-parameter GBM.")

Z=-2.0: exact S_T=67.70568745; Call payoff=0.00000000
Z=+0.0: exact S_T=101.00501671; Call payoff=1.00501671
Z=+2.0: exact S_T=150.68177851; Call payoff=50.68177851
Analytic E_Q[S_T]=103.04545340; discounted expectation=100.00000000
One exact terminal draw has no time-discretization error for this European payoff and constant-parameter GBM.


## ราคาเฉลี่ยต้องมาพร้อมความคลาดเคลื่อน

ให้ \(s_Y\) เป็น sample standard deviation ของ **discounted payoffs** ไม่ใช่ของราคาหุ้น

$$
s_Y^2=\frac{1}{N-1}\sum_{j=1}^N(Y_j-\widehat V_N)^2,
\qquad
\widehat{\operatorname{SE}}(\widehat V_N)=\frac{s_Y}{\sqrt N}.
$$

[Standard error](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#standard-error) วัดการแกว่งของค่าประมาณราคาเมื่อสุ่มใหม่ ช่วงความเชื่อมั่นโดยประมาณ 95% จาก Normal approximation คือ

$$
\widehat V_N\pm1.96\,\frac{s_Y}{\sqrt N}.
$$

เงื่อนไขสำคัญคือแต่ละรอบเป็นอิสระ มีความแปรปรวนจำกัด และจำนวนตัวอย่างเพียงพอสำหรับการประมาณนี้ หาก payoff ส่วนใหญ่เป็นศูนย์และมีเหตุการณ์หายากที่จ่ายสูง ช่วงที่คำนวณจากตัวอย่างน้อยอาจดูแคบเกินจริง

ช่วงนี้อธิบายความไม่แน่นอนของ **ค่าประมาณราคาในแบบจำลอง** ไม่ใช่ช่วงที่ราคาหุ้นจะอยู่ ไม่ใช่ขอบเขตขาดทุนสูงสุด และไม่รวม model error หากสุ่มซ้ำหลายชุด ช่วง 95% ก็ไม่จำเป็นต้องครอบ benchmark ทุกชุด

เมื่อเพิ่ม N เป็น 4 เท่า SE จะลดลงประมาณครึ่งหนึ่งในระยะยาว เช่น ถ้า SE≈0.14 ดอลลาร์ที่ 10,000 รอบ จะต้องใช้ประมาณ 40,000 รอบเพื่อให้เหลือ 0.07 ดอลลาร์ ค่า error ที่เกิดขึ้นจริงของการรันหนึ่งครั้งไม่จำเป็นต้องลดลงทุกครั้งที่เพิ่ม N



ข้อมูลจำลองด้วย Python และ seed ที่ระบุในภาพ จุดต่าง ๆ ใช้ตัวอย่างสะสมจากชุดเดียวกัน จึงพึ่งพากัน ไม่ใช่การทดลองอิสระหลายชุด

แยกความคลาดเคลื่อนอย่างน้อยสามชั้นก่อนตัดสินใจเพิ่มรอบคำนวณ

| สิ่งที่คลาดเคลื่อน | เกิดจากอะไร | วิธีตรวจหรือปรับ |
|---|---|---|
| Sampling error | ใช้ตัวอย่างสุ่มจำนวนจำกัด | รายงาน SE/CI เพิ่ม N หรือใช้ variance reduction ที่ถูกต้อง |
| Discretization / monitoring error | แทนเส้นทางต่อเนื่องด้วย step หรือวันสังเกต | ลด time step ใช้ exact update หรือแก้การเฝ้า barrier ตามลักษณะสัญญา |
| Model error | กติกาและพารามิเตอร์ไม่อธิบายความเสี่ยงที่ต้องการ | ตรวจข้อมูล สมมติฐานและแบบจำลองทางเลือก |

การสุ่มเพิ่มลดชั้นแรก การลด time step แก้ชั้นที่สอง ทั้งสองอย่างไม่ทำให้ volatility คงที่กลายเป็นสมมติฐานที่เหมาะกับทุกตลาด

In [4]:
from statistics import mean, stdev
mc = monte_carlo(paths=20000,seed=2530401,checkpoints=[100,500,1000,5000])
# Recompute independently from the same normal draws to check online variance.
rng = random.Random(mc["seed"])
discounted_payoffs = [math.exp(-r*T)*payoff(S0*math.exp((r-.5*sigma*sigma)*T+sigma*math.sqrt(T)*rng.gauss(0,1))) for _ in range(mc["paths"])]
close(mc["price"],mean(discounted_payoffs))
close(mc["sd"],stdev(discounted_payoffs))
close(mc["se"],stdev(discounted_payoffs)/math.sqrt(len(discounted_payoffs)))
close(mc["upper"]-mc["price"],1.96*mc["se"])
print(f"Python seed={mc['seed']}; N={mc['paths']:,}; iid exact terminal draws")
print(f"MC={mc['price']:.10f}; sample SD={mc['sd']:.10f}; sample SE={mc['se']:.10f}")
print(f"Approximate 95% CI=[{mc['lower']:.10f},{mc['upper']:.10f}]")
print(f"Analytic error={mc['price']-mc['reference']:+.10f}; error/SE={(mc['price']-mc['reference'])/mc['se']:+.4f}")
print("A pointwise approximate CI need not cover the true value in every run. Model error is excluded.")

Python seed=2530401; N=20,000; iid exact terminal draws
MC=9.3743990008; sample SD=14.0374836002; sample SE=0.0992599984
Approximate 95% CI=[9.1798494039,9.5689485978]
Analytic error=-0.0390043830; error/SE=-0.3930
A pointwise approximate CI need not cover the true value in every run. Model error is excluded.


## ทดลอง: เพิ่มจำนวนรอบแล้วราคาเปลี่ยนอย่างไร

เริ่มจาก Call ที่ค่าเดิม เพิ่ม N โดยคง seed แล้วสังเกตราคาและ SE จากนั้นกดสุ่มชุดใหม่เพื่อดูการเปลี่ยนแปลงข้ามชุด กราฟใช้แกน log ของ N เพื่อให้มองเห็นทั้งหลักร้อยและหลักหมื่น

ห้องทดลองจำลอง \(S_T\) แบบ exact และใช้ pseudo-random generator เดิมของซีรีส์กับ Box–Muller เพื่อการศึกษา Notebook ใช้ตัวสุ่มของ Python จึงไม่จำเป็นต้องได้ราคา Monte Carlo เท่ากับเว็บทุกหลัก แม้ใส่ seed เลขเดียวกัน

In [5]:
print("N       MC price        sample SE       approximate 95% interval")
for row in mc["checkpoints"]:
    print(f"{row['paths']:6d} {row['price']:14.8f} {row['se']:14.8f}  [{row['lower']:.8f},{row['upper']:.8f}]")
small = next(row for row in mc["checkpoints"] if row["paths"]==5000)
print(f"20,000 / 5,000 = 4; measured SE ratio={small['se']/mc['se']:.5f}, near sqrt(4)=2.")
print("All rows use nested prefixes; the errors do not have to decrease monotonically.")
print("Website and Python RNG implementations differ; identical seeds need not give identical prices.")

N       MC price        sample SE       approximate 95% interval
   100     8.22707243     1.30697449  [5.66540244,10.78874243]
   500     9.07486777     0.62646169  [7.84700286,10.30273268]
  1000     9.53694035     0.46533891  [8.62487609,10.44900461]
  5000     9.48499835     0.19859214  [9.09575774,9.87423895]
 20000     9.37439900     0.09926000  [9.17984940,9.56894860]
20,000 / 5,000 = 4; measured SE ratio=2.00073, near sqrt(4)=2.
All rows use nested prefixes; the errors do not have to decrease monotonically.
Website and Python RNG implementations differ; identical seeds need not give identical prices.


## อีกมุมหนึ่ง: แทนอนุพันธ์ด้วยค่าบนกริด

[Finite difference](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#finite-difference) เริ่มจากการเก็บค่า Option บนจุดราคาและเวลา แทนการสุ่มเส้นทางจำนวนมาก ให้

$$
S_i=i\Delta S,\quad \Delta S=\frac{S_{\max}}{M},\quad
\tau_k=k\Delta\tau,\quad \Delta\tau=\frac{T}{L},
\quad U_i^k=V(S_i,T-\tau_k).
$$

i วิ่งจาก 0 ถึง M และ k วิ่งจาก 0 ถึง L เราใช้ \(\tau=T-t\) คือ **เวลาที่เหลือถึงวันหมดอายุ** เมื่อเริ่มที่ τ=0 เรารู้ payoff อยู่แล้ว จากนั้นเพิ่ม τ จนถึง T เพื่อหาค่าที่วันนี้ ตัวแปรเวลาในคอมพิวเตอร์จึงเดินจาก payoff กลับมาหาปัจจุบันของสัญญา

บนกริดราคาที่ห่างเท่ากัน เราประมาณ Delta ด้วย central difference และ Gamma ด้วย second difference

$$
\Delta_i^k\approx\frac{U_{i+1}^k-U_{i-1}^k}{2\Delta S},
\qquad
\Gamma_i^k\approx\frac{U_{i+1}^k-2U_i^k+U_{i-1}^k}{(\Delta S)^2}.
$$

Delta บอกความชันของราคา Option ต่อราคาหุ้น ส่วน Gamma บอกว่าความชันเปลี่ยนเร็วเพียงใด ตัวอย่าง \(\Delta S=5\) และค่า Option ที่ราคาสามจุดเป็น 7, 10, 14 จะได้ Delta≈0.7 และ Gamma≈0.04 ต่อดอลลาร์ ตัวเลขสามจุดนี้เป็นตัวอย่างสำหรับฝึกสูตร ไม่ใช่ผลลัพธ์กริดจริง

Forward difference ใช้ \((U_{i+1}-U_i)/\Delta S\) ส่วน backward difference ใช้ \((U_i-U_{i-1})/\Delta S\) เมื่อคำตอบเรียบพอ central difference ของ Delta และ Gamma มี truncation error อันดับ \(O((\Delta S)^2)\) ขณะที่ forward/backward difference ของ Delta มีอันดับ \(O(\Delta S)\) แต่ใกล้ขอบกริดหรือเมื่อ drift เด่นกว่า diffusion การเลือกด้านเดียวอาจเหมาะกว่าด้านเสถียรภาพ

สำหรับ Theta ต้องระวังเครื่องหมาย เพราะ \(\partial_tV=-\partial_\tau U\)

$$
\Theta(S_i,t)\approx\frac{U_i^k-U_i^{k+1}}{\Delta\tau}.
$$

In [6]:
# Smooth cubic: verify the orders without option-payoff nonsmoothness.
S=100.
f=lambda s:s**3
print("dS     forward error     backward error    central error     gamma error")
for dS in [10.,5.,2.5]:
    forward=(f(S+dS)-f(S))/dS
    backward=(f(S)-f(S-dS))/dS
    central=(f(S+dS)-f(S-dS))/(2*dS)
    gamma=(f(S+dS)-2*f(S)+f(S-dS))/dS**2
    close(central-3*S*S,dS*dS)
    close(gamma,6*S)
    print(f"{dS:4.1f} {forward-3*S*S:17.5f} {backward-3*S*S:17.5f} {central-3*S*S:16.5f} {gamma-6*S:15.5f}")
print("For this smooth cubic, central delta error is exactly dS^2. This does not assume smoothness at the Call payoff kink.")

dS     forward error     backward error    central error     gamma error
10.0        3100.00000       -2900.00000        100.00000         0.00000
 5.0        1525.00000       -1475.00000         25.00000         0.00000
 2.5         756.25000        -743.75000          6.25000         0.00000
For this smooth cubic, central delta error is exactly dS^2. This does not assume smoothness at the Call payoff kink.


## จาก Black–Scholes PDE สู่สูตรสามจุด

เขียน PDE ด้วยเวลา τ จะได้

$$
\frac{\partial U}{\partial\tau}
=\frac12\sigma^2S^2\frac{\partial^2U}{\partial S^2}
+rS\frac{\partial U}{\partial S}-rU.
$$

แทนด้านซ้ายด้วย \((U_i^{k+1}-U_i^k)/\Delta\tau\) และใช้อันดับเวลา k สำหรับทุกพจน์ด้านขวา จะได้วิธี **explicit**: ค่าชั้นใหม่คำนวณจากค่าที่รู้แล้วสามจุด

$$
U_i^{k+1}=a_iU_{i-1}^k+b_iU_i^k+c_iU_{i+1}^k,
$$

$$
\begin{aligned}
a_i&=\frac{\Delta\tau}{2}(\sigma^2i^2-ri),\\
b_i&=1-\Delta\tau(\sigma^2i^2+r),\\
c_i&=\frac{\Delta\tau}{2}(\sigma^2i^2+ri).
\end{aligned}
$$

เมื่อ \(S_{\max}=400\), M=80, L=1,000 จะได้ ΔS=5 ดอลลาร์ และ Δτ=0.001 ปี ที่ S=100 หรือ i=20 สัมประสิทธิ์เป็น **a=0.0077, b=0.98397, c=0.0083** ผลรวมเท่ากับ 0.99997 หรือ \(1-r\Delta\tau\)

ที่วันหมดอายุ Call มีค่า 0, 0, 5 ดอลลาร์บน S=95,100,105 ดังนั้นการเดินกลับจาก expiry หนึ่ง step ให้ค่าที่ S=100 เป็น \(0.0077(0)+0.98397(0)+0.0083(5)=0.0415\) ดอลลาร์ แล้วใช้ทั้งแถวที่คำนวณได้ทำ step ถัดไป

ต้องเขียนค่าชั้นใหม่ลงอาร์เรย์อีกชุดก่อนสลับกับชั้นเดิม หากเขียนทับค่าทีละจุดแล้วใช้จุดที่เพิ่งอัปเดตในการคำนวณจุดถัดไป เราจะเปลี่ยนวิธีเชิงตัวเลขโดยไม่ตั้งใจ

สูตรนี้คล้ายการคิดย้อนกลับบนต้นไม้ แต่ a,b,c **ไม่ใช่ probabilities ที่รวมได้หนึ่ง** โดยตรง มีส่วนคิดลดรวมอยู่ด้วย และบางการตั้งค่าทำให้สัมประสิทธิ์ติดลบได้

In [7]:
coefficients = explicit_coefficients()
dt,dS,i = 1/1000,400/80,20
a,b,c = coefficients[i-1]
close(a,.0077);close(b,.98397);close(c,.0083)
for weights in coefficients:
    assert min(weights)>=0
    close(sum(weights),1-r*dt)
first_step = a*payoff((i-1)*dS)+b*payoff(i*dS)+c*payoff((i+1)*dS)
close(first_step,.0415)
print(f"At S=i*dS=100: a={a:.8f}, b={b:.8f}, c={c:.8f}")
print(f"First explicit update from payoff: V(tau=0.001,S=100)={first_step:.8f}")
print(f"Weight sum={a+b+c:.8f}=1-r*dt, not 1; these are not normalized transition probabilities.")

At S=i*dS=100: a=0.00770000, b=0.98397000, c=0.00830000
First explicit update from payoff: V(tau=0.001,S=100)=0.04150000
Weight sum=0.99997000=1-r*dt, not 1; these are not normalized transition probabilities.


## ต้องรู้ payoff และค่าที่ขอบก่อนเริ่ม

ที่ τ=0 เรากำหนดทุกจุดเป็น payoff

$$
U_i^0=\max(S_i-K,0)\quad\text{(Call)},\qquad
U_i^0=\max(K-S_i,0)\quad\text{(Put)}.
$$

นี่คือ terminal condition ในเวลา t และเป็น initial condition ในเวลา τ ส่วน [boundary conditions](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#boundary-condition) คือค่าที่ปลายกริดราคาซึ่งต้องกำหนด **ทุกชั้นเวลา**

| สัญญา | ขอบ S=0 | ขอบ S=Smax ที่สูงเพียงพอ |
|---|---|---|
| European Call | \(U(0,\tau)=0\) | \(U(S_{\max},\tau)\approx S_{\max}-Ke^{-r\tau}\) |
| European Put | \(U(0,\tau)=Ke^{-r\tau}\) | \(U(S_{\max},\tau)\approx0\) |

หุ้นที่เริ่มเป็นศูนย์จะคงเป็นศูนย์ใน GBM: Call จึงไม่มี payoff ส่วน Put จ่าย K เมื่อหมดอายุและมีค่าปัจจุบัน \(Ke^{-r\tau}\) สำหรับ Call ที่ S สูงมาก มูลค่าเข้าใกล้หุ้นหนึ่งหน่วยลบมูลค่าปัจจุบันของ K

Smax เป็นการแทนช่วงราคาอนันต์ด้วยขอบจำกัด จึงเกิด [domain truncation error](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#domain-truncation) ได้ หากขยับ Smax ออกแล้วราคาที่สนใจเปลี่ยนมาก แสดงว่าขอบเดิมใกล้เกินไป แต่ต้องคง ΔS ให้ใกล้เดิมด้วย มิฉะนั้นเราจะเปลี่ยนทั้งตำแหน่งขอบและความละเอียดพร้อมกัน

เอกสารยังเสนอขอบบนแบบ \(\Gamma\approx0\) หรือ \(U_M^k=2U_{M-1}^k-U_{M-2}^k\) สำหรับ payoff ที่เกือบเป็นเส้นตรงเมื่อ S สูง เป็นทางเลือกแบบ linear extrapolation ซึ่งต้องตรวจความเหมาะสมแยกต่างหาก ห้องทดลองนี้ใช้ค่าขอบ Call/Put ในตารางโดยตรง

In [8]:
for tau in [0.,.5,1.]:
    call_low,call_high = 0.,400-K*math.exp(-r*tau)
    put_low,put_high = K*math.exp(-r*tau),0.
    close(black_scholes(S=0,T=tau),call_low)
    close(black_scholes(S=0,T=tau,kind="put"),put_low)
    print(f"tau={tau:.1f}: Call boundaries=({call_low:.8f},{call_high:.8f}); Put boundaries=({put_low:.8f},{put_high:.8f})")
print("The high-S boundary at Smax is an asymptotic approximation; Smax is finite, not infinity.")
for kind in ["call","put"]:
    result=finite_difference(kind=kind)
    close(result['values'][0],0 if kind=='call' else K*math.exp(-r*T))
    close(result['values'][-1],400-K*math.exp(-r*T) if kind=='call' else 0)
    assert min(result['values'])>=0
    differences=[b-a for a,b in zip(result['values'],result['values'][1:])]
    assert (min(differences)>=-1e-12) if kind=='call' else (max(differences)<=1e-12)
    print(f"{kind}: nonnegative grid and expected monotonicity checked.")

tau=0.0: Call boundaries=(0.00000000,300.00000000); Put boundaries=(100.00000000,0.00000000)
tau=0.5: Call boundaries=(0.00000000,301.48880604); Put boundaries=(98.51119396,0.00000000)
tau=1.0: Call boundaries=(0.00000000,302.95544665); Put boundaries=(97.04455335,0.00000000)
The high-S boundary at Smax is an asymptotic approximation; Smax is finite, not infinity.
call: nonnegative grid and expected monotonicity checked.
put: nonnegative grid and expected monotonicity checked.


## กริดละเอียดขึ้นอาจทำให้วิธี explicit ใช้ไม่ได้

การใช้ time step ใหญ่เกินไปอาจขยายความผิดพลาดจนคำตอบสั่นหรือระเบิด สำหรับตัวอย่างที่ r≥0 เราใช้เกณฑ์เพียงพอที่ตรวจง่าย: **สัมประสิทธิ์ a,b,c ของทุกจุดภายในต้องไม่ติดลบ** ทำให้แต่ละ step ไม่ขยายค่าสูงสุดจากการถ่วงน้ำหนักภายในกริด เพราะผลรวมไม่เกินหนึ่ง

เงื่อนไขของ b ให้ข้อจำกัด

$$
\Delta\tau\leq\frac{1}{\sigma^2(M-1)^2+r}.
$$

ที่ค่าตั้งต้น M=80 ต้องมี L อย่างน้อย **250 step** เมื่อ T=1 ปี ดังนั้น 1,000 step ผ่านเงื่อนไขนี้ แต่ 100 step ไม่ผ่าน หากเพิ่ม M ประมาณสองเท่า time step ที่ยอมให้ใช้จะเล็กลงประมาณสี่เท่า งานคำนวณกริดหนึ่งมิติจึงเพิ่มเร็วกว่าจำนวนจุดราคาเพียงอย่างเดียว

ยังมีอีกเงื่อนไขที่การลด time step แก้ไม่ได้: \(a_i\geq0\) ต้องการ \(\sigma^2i\geq r\) ตัวอย่าง r=5%, σ=20%, i=1 จะได้ \(\sigma^2i=0.04<0.05\) ทำให้ a ติดลบสำหรับทุก Δτ>0 ทางเลือกหนึ่งคือเปลี่ยนวิธีประมาณ drift หรือใช้กริด/วิธีที่เหมาะสม ไม่ใช่เพิ่ม L ไปเรื่อย ๆ

การไม่ผ่านเกณฑ์นี้ไม่ได้พิสูจน์ว่าทุกการรันจะระเบิด แต่ **ไม่ผ่านเงื่อนไขรับรองที่ห้องทดลองเลือกใช้** จึงหยุดแสดงราคาและอธิบายเหตุผล เราไม่ต้องรอให้เห็นตัวเลขผิดปกติก่อนตรวจ scheme

[เสถียรภาพ](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#numerical-stability)และความแม่นยำก็เป็นคนละเรื่อง กริดที่ผ่านเกณฑ์อาจยังหยาบ ขอบราคาอาจใกล้เกินไป หรือ payoff อาจไม่เรียบที่ strike จึงต้องตรวจ convergence ต่อ

In [9]:
M,N=80,1000
max_dt=1/(sigma*sigma*(M-1)**2+r)
minimum_steps=math.ceil(T/max_dt)
print(f"For M={M}: dt <= {max_dt:.9f}; at least {minimum_steps} steps for nonnegative b_i.")
print(f"Default dt={T/N:.9f}; all weights nonnegative, minimum={min(min(row) for row in coefficients):.9f}")
for label,params in [("Too few time steps",{"steps":100}),("Central drift has a_1<0",{"r":.08})]:
    try:
        finite_difference(**params)
    except ValueError as error:
        print(label+": rejected — "+str(error))
    else:
        raise AssertionError("Unsafe configuration was accepted")
print("Increasing N repairs the first case; r>sigma^2 at i=1 needs a different drift stencil.")

For M=80: dt <= 0.004005287; at least 250 steps for nonnegative b_i.
Default dt=0.001000000; all weights nonnegative, minimum=0.000005000
Too few time steps: rejected — Unsafe explicit time step: negative coefficient; increase steps
Central drift has a_1<0: rejected — Unsafe central drift: a_1<0; more time steps do not fix it
Increasing N repairs the first case; r>sigma^2 at i=1 needs a different drift stencil.


## ทดลอง: เปลี่ยนกริดและตรวจราคาก่อนใช้

เริ่มจาก M=80, L=1,000 เทียบราคาและ Greeks กับ Black–Scholes จากนั้นลด L เป็น 100 เพื่อดูการตรวจเสถียรภาพ ลองเพิ่ม M โดยคง L แล้วใช้ปุ่มปรับจำนวน time step ให้ผ่านเกณฑ์ หากยังไม่ผ่าน ให้ดูเงื่อนไขด้าน drift ด้วย



กริดและเส้นราคาเกิดจากการคำนวณจริงภายใต้สมมติฐานตัวอย่าง กราฟ error แยกไว้เพื่อให้เห็นความต่างที่เส้นราคาหลักอาจทับกันจนดูไม่ออก

In [10]:
fd_call=finite_difference()
fd_put=finite_difference(kind="put")
close(fd_call["price"],9.352555601587657)
close(fd_put["price"],6.397065285525917)
for kind,result in [("Call",fd_call),("Put",fd_put)]:
    print(f"{kind}: FD={result['price']:.10f}; analytic={result['reference']:.10f}; error={result['price']-result['reference']:+.10f}")
parity_error=fd_call['price']-fd_put['price']-(S0-K*math.exp(-r*T))
print(f"Discrete put-call parity residual={parity_error:+.10f}; explicit discounting has time-discretization error.")
assert abs(parity_error)<.00005
print("Price at non-node S uses linear interpolation; reported Greeks name the nearest interior node.")

Call: FD=9.3525556016; analytic=9.4134033839; error=-0.0608477823
Put: FD=6.3970652855; analytic=6.4579567387; error=-0.0608914532
Discrete put-call parity residual=+0.0000436709; explicit discounting has time-discretization error.
Price at non-node S uses linear interpolation; reported Greeks name the nearest interior node.


## ตรวจทั้งราคา Greeks และการลู่เข้า

เมื่อมีคำตอบบนกริด เราคำนวณ Delta และ Gamma จากจุดข้างเคียงได้ ส่วน Theta ที่วันนี้ประมาณจากสองชั้นเวลาสุดท้ายด้วย \((U_i^{L-1}-U_i^L)/\Delta\tau\) ซึ่งมีหน่วยดอลลาร์ต่อปี ห้องทดลองใช้ linear interpolation หาก S ไม่ตรงจุดกริด ส่วน Greeks ที่ขอบไม่แสดง เพราะสูตรกลางต้องมีเพื่อนบ้านสองด้าน

Gamma หักล้างค่าราคาที่อยู่ใกล้กันแล้วหารด้วย \((\Delta S)^2\) จึงไวต่อความผิดพลาดของกริด ราคาใกล้ benchmark ไม่ได้แปลว่า Greeks แม่นในระดับเดียวกัน โดยเฉพาะใกล้ strike และใกล้หมดอายุ



ตัวอย่าง convergence ภายใต้ Smax คงที่และ strike ตรงกับจุดกริดแต่ละชุด การลด error ในตัวอย่างนี้ไม่ใช่การรับประกันรูปแบบเดียวกันสำหรับทุก payoff และทุกกริด

การตรวจที่มีประโยชน์คือ

1. **ตรวจโจทย์ที่รู้คำตอบ:** เทียบ Call/Put และ Greeks กับสูตร Black–Scholes
2. **ลด Δτ โดยคง ΔS:** ดูผลจากการแบ่งเวลา ภายใต้กริดที่ผ่านเกณฑ์
3. **ลด ΔS พร้อมรักษาเสถียรภาพ:** ดูว่าคำตอบเข้าใกล้กันหรือไม่
4. **ขยาย Smax โดยคง ΔS:** ตรวจผลของขอบจำกัด
5. **ตรวจสมบัติของราคา:** เช่น ไม่ติดลบ ขอบล่างตาม no-arbitrage และ \(C-P=S-Ke^{-rT}\) โดยคำนึงถึง numerical tolerance

การประมาณ PDE มี formal truncation error \(O(\Delta\tau+(\Delta S)^2)\) เมื่อคำตอบเรียบพอ แต่ payoff ของ Call/Put มีมุมที่ strike และเงื่อนไขขอบเป็นอีกแหล่ง error จึงไม่ควรสรุป global accuracy จากอันดับของสูตรเพียงอย่างเดียว

In [11]:
reference_greeks=black_scholes_greeks()
print(f"Greeks evaluated at S={fd_call['greek_spot']:.1f}; theta per year in calendar time t:")
for key in ["delta","gamma","theta"]:
    print(f"{key:6s}: FD={fd_call[key]:+.10f}; analytic={reference_greeks[key]:+.10f}")
rows=convergence_rows()
print("M       N       dS          dt           FD price       absolute error")
for row in rows:
    print(f"{row['intervals']:3d} {row['steps']:7d} {row['dS']:8.3f} {row['dt']:11.7f} {row['price']:16.10f} {row['error']:17.10f}")
assert rows[-1]['error']<.004
assert all(a['error']>b['error'] for a,b in zip(rows,rows[1:]))
print("Observed error ratios:",[round(a['error']/b['error'],5) for a,b in zip(rows,rows[1:])])
print("dt/dS^2 remains fixed and the strike is on each grid. This example is not a general convergence proof.")
print("All assertions passed.")

Greeks evaluated at S=100.0; theta per year in calendar time t:
delta : FD=+0.5973438741; analytic=+0.5987063257
gamma : FD=+0.0195012808; analytic=+0.0193334058
theta : FD=-5.4138308979; analytic=-5.3803980436
M       N       dS          dt           FD price       absolute error
 40     250   10.000   0.0040000     9.1597212497      0.2536821341
 80    1000    5.000   0.0010000     9.3525556016      0.0608477823
160    4000    2.500   0.0002500     9.3983241078      0.0150792761
320   16000    1.250   0.0000625     9.4096415588      0.0037618251
Observed error ratios: [4.16913, 4.03519, 4.0085]
dt/dS^2 remains fixed and the strike is on each grid. This example is not a general convergence proof.
All assertions passed.


## เลือกวิธีให้เข้ากับสัญญา

| ลักษณะโจทย์ | Monte Carlo | Finite difference |
|---|---|---|
| European payoff ที่รู้การแจกแจงปลายทาง | จำลองปลายทางได้โดยตรง | ได้ราคาบนกริดหลายค่า S ในการรันเดียว |
| หลายสินทรัพย์ | เพิ่มมิติในช็อกและ covariance ได้ | จำนวนจุดกริดโตเร็วตามจำนวนมิติ |
| ขึ้นกับเส้นทาง | เก็บค่าที่ต้องใช้ระหว่างจำลองได้ | อาจต้องเพิ่ม state variable เช่นค่าเฉลี่ยสะสม |
| Greeks | ต้องเพิ่มวิธี เช่น common random numbers หรือ pathwise differentiation ตามเงื่อนไข | หา Delta/Gamma จากกริดได้ แต่ต้องตรวจความไวต่อความละเอียด |
| Early exercise | ค่าเฉลี่ย payoff ที่ expiry อย่างเดียวไม่พอ ต้องประเมิน continuation value | เปรียบเทียบ continuation กับ exercise value ที่แต่ละ step ได้ |

สำหรับ American Option แนวคิดบนกริดคือเปรียบเทียบค่าถือสัญญาต่อกับค่าที่ได้จากการใช้สิทธิทันที: \(U_i^{k+1}=\max(U_{i,\mathrm{continue}}^{k+1},g(S_i))\) ต้องกำหนดขอบและ scheme ให้สอดคล้องกับปัญหา American ด้วย ห้องทดลองในหน้านี้และ Notebook ใช้ **European Call/Put เท่านั้น**

วิธี explicit เริ่มเขียนและตรวจได้ง่าย แต่ติดข้อจำกัด time step วิธี implicit และ Crank–Nicolson เป็นหัวข้อต่อยอดที่เปลี่ยนการแก้ชั้นเวลา และมีข้อพิจารณาด้านความแม่นยำ/การสั่นใกล้ payoff ของตนเอง

## ลองตอบก่อนเปิด Notebook

**1.** Monte Carlo ให้ SE=0.12 ดอลลาร์จาก 10,000 รอบ ต้องใช้ประมาณกี่รอบให้ SE เหลือ 0.03 ดอลลาร์?

**2.** ถ้าเพิ่มจำนวนช่วงราคา M เป็นสองเท่า แต่คง L เดิม ทำไมกริด explicit ที่เคยผ่านเกณฑ์อาจไม่ผ่าน?

**3.** ทำไมการใช้ exact GBM ที่วันต้นและวันปลายสัญญายังไม่พอสำหรับ continuous barrier?

**4.** ที่ S=0 ทำไม European Put มีค่า \(Ke^{-r\tau}\) แทน K และเหตุใดเงื่อนไขนี้จึงไม่ใช้กับ American Put แบบตรง ๆ?

**เปิดแนวคำตอบ**

1. ต้องลด SE เป็นหนึ่งในสี่ จึงใช้ N ประมาณ 16 เท่า หรือ 160,000 รอบ เมื่อความแปรปรวนของ discounted payoff คงเดิม
2. พจน์ \(\sigma^2i^2\Delta\tau\) ที่ปลายกริดโตขึ้นประมาณสี่เท่า b อาจติดลบ ต้องลด Δτ ประมาณสี่เท่า และตรวจเงื่อนไข drift แยกด้วย
3. สองจุดไม่บอกว่าราคาระหว่างทางเคยข้าม barrier หรือไม่ ต้องจัดการการเฝ้าระหว่างจุด เช่นเพิ่มการสังเกตหรือใช้วิธี Brownian bridge ที่เหมาะสม
4. European Put จ่าย K ในอนาคตเมื่อหุ้นคงเป็นศูนย์ จึงต้องคิดลด สำหรับ American Put ผู้ถือมีสิทธิใช้ก่อนหมดอายุ ต้องพิจารณาการใช้สิทธิทันทีร่วมด้วย ภายใต้ r≥0 ที่ S=0 การรับ K ทันทีไม่ด้อยกว่าการรอ

[ดาวน์โหลด Python Notebook](numerical-methods.ipynb) เพื่อรัน Monte Carlo ตรวจ SE และเปรียบเทียบ finite difference หลายกริดกับ Black–Scholes โค้ดใช้ Python standard library และฝังภาพประกอบไว้ในไฟล์

## แหล่งที่มาและขอบเขต

เรียบเรียงจากเอกสาร **Introduction to Numerical Methods** ของหลักสูตร Certificate in Quantitative Finance ในไฟล์ *JA253.4 Notes.pdf* ที่ผู้ใช้ให้มา 62 หน้า: risk-neutral Monte Carlo หน้า 4–17, กริดและอนุพันธ์หน้า 18–34, payoff และ explicit scheme หน้า 35–48, boundary conditions หน้า 49–58 และการเลือกวิธีหน้า 59–62

เนื้อหานี้อธิบายใหม่เป็นภาษาไทย เพิ่มตัวอย่างคำนวณ ห้องทดลองและการตรวจเสถียรภาพ ไม่ได้เผยแพร่ไฟล์ PDF ต้นฉบับหรือภาพหน้าสไลด์ จุดขยายความคือความต่างของ weak/strong error, exact update กับการสังเกตเส้นทาง, ช่วงความเชื่อมั่นของ Monte Carlo และข้อจำกัดสัมประสิทธิ์ central difference

เอกสารประกอบสำหรับตรวจสูตร: Mike Giles, University of Oxford, [Monte Carlo Lecture 1](https://people.maths.ox.ac.uk/gilesm/mc/mc/lec1.pdf) สำหรับ terminal GBM และ Box–Muller และ [Lecture 9](https://people.maths.ox.ac.uk/gilesm/mc/mc/lec9.pdf) สำหรับ strong/weak convergence

ราคาและกราฟทุกชุดเป็นตัวอย่างสมมติภายใต้แบบจำลอง ไม่ใช่ market quote หรือการสอบเทียบกับตลาด รายละเอียดวิธีคำนวณและแหล่งที่มาอยู่ใน [provenance](https://nutdnuy.github.io/quantitative-finance-notes/data/numerical-methods-provenance.json)